In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn



In [2]:
path = 'data/raw/consolidated_traffic_data.csv'
df = pd.read_csv(path)

In [4]:
print(f"Rows = {df.shape[0]}, Columns = {df.shape[1]}")


Rows = 59706, Columns = 24


In [7]:
print(df.head())

      duration   total_fiat   total_biat  min_fiat  min_biat     max_fiat  \
0  119994459.0  119994459.0  119970125.0       0.0       0.0    1399330.0   
1  143704195.0  143704195.0   42106019.0       0.0       0.0  101572528.0   
2  119994980.0  119994980.0  119966891.0       7.0       0.0     819852.0   
3  119292282.0  119292282.0  119260024.0       3.0       0.0     912990.0   
4  119772966.0  119772917.0  119772939.0       4.0       0.0    2894671.0   

    max_biat     mean_fiat    mean_biat  flowPktsPerSecond  ...  \
0  1211310.0   2348.136257   513.267526        2373.792943  ...   
1   854818.0   6986.105737   636.898837         603.204381  ...   
2   819602.0  10201.919742  2341.411304         525.030297  ...   
3   866900.0  11947.148923  3354.712349         381.726288  ...   
4  2923147.0  18210.873803  3644.280989         329.331412  ...   

     std_flowiat   min_active  mean_active   max_active    std_active  \
0    9107.759178    1753114.0   11300000.0   39438693.0  1.60

In [3]:
target_col = 'traffic_type'
print(f"Target Column: {target_col}\n")
print(df[target_col].value_counts())

Target Column: traffic_type

traffic_type
BROWSING         10000
VPN-BROWSING     10000
VOIP              6485
VPN-VOIP          5576
VPN-FT            4704
P2P               4000
FT                3975
VPN-P2P           3415
VPN-CHAT          2839
CHAT              2505
VPN-MAIL          2444
MAIL              1364
STREAMING         1284
VPN-STREAMING     1115
Name: count, dtype: int64


In [4]:
qos_map = {
    'VOIP': 0, 'VPN-VOIP': 0,                          
    'STREAMING': 1, 'VPN-STREAMING': 1,               
    'CHAT': 1, 'VPN-CHAT': 1,
    'BROWSING': 2, 'VPN-BROWSING': 2,                 
    'MAIL': 2, 'VPN-MAIL': 2,
    'FT': 3, 'VPN-FT': 3,                             
    'P2P': 3, 'VPN-P2P': 3
}

In [5]:
df['qos_class'] = df['traffic_type'].map(qos_map)

In [6]:
print("Samples per QoS Queue:")
print(df['qos_class'].value_counts().sort_index())

Samples per QoS Queue:
qos_class
0    12061
1     7743
2    23808
3    16094
Name: count, dtype: int64


In [7]:
drop_cols = ['traffic_type', 'qos_class']

for col in df.columns:
    if df[col].dtype == 'object' and col not in drop_cols:
        drop_cols.append(col)

X = df.drop(columns=drop_cols)
y = df['qos_class'].values

X = X.replace([np.inf, -np.inf], np.nan).fillna(0)

print(f"Features extracted: {X.shape[1]} input dimensions across {X.shape[0]} samples")

Features extracted: 23 input dimensions across 59706 samples


In [8]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X.values, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [10]:
print(f"training samples: {X_train.shape[0]} , test samples: {X_test.shape[0]}")

training samples: 47764 , test samples: 11942


In [11]:
import torch
from torch.utils.data import DataLoader, TensorDataset

X_train_tensor = torch.tensor(X_train, dtype=torch.float32).unsqueeze(1)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)

X_test_tensor = torch.tensor(X_test, dtype=torch.float32).unsqueeze(1)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=128, shuffle=True)
test_loader = DataLoader(TensorDataset(X_test_tensor, y_test_tensor), batch_size=256, shuffle=False)

print(f"X_train tensor shape: {X_train_tensor.shape}")
print(f"Ready for Conv1d with {X_train_tensor.shape[2]} sequence length / feature nodes.")

X_train tensor shape: torch.Size([47764, 1, 23])
Ready for Conv1d with 23 sequence length / feature nodes.


In [18]:
import torch.nn as nn
import torch.nn.functional as F

class cnn1d(nn.Module):
    def __init__(self,num_classes = 4):
        super(cnn1d,self).__init__()

        self.conv1 = nn.Conv1d(in_channels=1, out_channels=32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm1d(32)

        self.conv2 = nn.Conv1d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm1d(64)

        self.pool = nn.MaxPool1d(kernel_size=2)
        self.dropout = nn.Dropout(0.3)

        self.fc1 = nn.Linear(64*5,128)
        self.fc2 = nn.Linear(128,num_classes)

    def forward(self,x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))

        x = x.view(x.size(0),-1)
        x = self.dropout(F.relu(self.fc1(x)))
        x = self.fc2(x)
        return x

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = cnn1d(num_classes = 4).to(device)

dummy_in = torch.randn(2,1,23).to(device)
op = model(dummy_in)

print(f"Hardware device: {device}")
print(f"output logits the shape :{op.shape} ")


Hardware device: cuda
output logits the shape :torch.Size([2, 4]) 


In [ ]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr = 0.001)

epochs = 15

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0
    for inputs, labels in train_loader:
        inputs,labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)

        loss = criterion(outputs,labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs.data,1)
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()
    train_acc = 100*correct_train/total_train

    model.eval()
    correct_test = 0
    total_test = 0

    with torch.no_grad():
        for inputs,labels in test_loader:
            inputs,labels = inputs.to(device), labels.to(device)

            outputs = model(inputs)

            _, predicted = torch.max(outputs.data,1)
            total_test += labels.size(0)
            correct_test += (predicted == labels).sum().item()
    test_acc = 100*correct_test/total_test
    print(f"Epoch [{epoch+1:2d}/{epochs}], Loss: {running_loss/len(train_loader):.4f}")
    print(f"Train accuracy: {train_acc: .2f}% Test accuracy: {test_acc: .2f}%")

_IncompleteInputError: incomplete input (791671746.py, line 8)